# example / wasm / saturate

**Intent**
- Estimates `saturate_out` from `saturate_in`, and must:
  - Give the fitted height times the tanh of the fitted rate times the input.
  - Show a model that binds a host `tanh`, with a golden tolerance of 1e-12.
  - Hold the rmse on `validation` and `backtest` to 0.5.
  - Stay a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on the curve 4 tanh(0.5x) with fixed noise, from x of -2.5 to 2.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is seven inputs across both signs and the saturated tail, then a request with the input missing.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import math

from scipy.optimize import curve_fit

from mlmodels import setup_workspace
from mlmodels.maths import tanh

ws = setup_workspace("example/wasm/saturate")


ws.outputs[0] = ws.outputs[0]

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [first + 0.5 * step for step in range(count)]
    return {ws.inputs[0]: xs, ws.outputs[0]: [round(4.0 * math.tanh(0.5 * x) + NOISE[int(round(2 * x)) % 10], 4) for x in xs]}


collect_run(ws, 1000, label="Ten labelled points on the curve 4 tanh(0.5x) with fixed noise", name="wasm_saturate_training", source="builder", build=lambda: labelled(-2.5, 10))
collect_run(ws, 1000, label="Five more points on the curve", name="wasm_saturate_validation", source="builder", build=lambda: labelled(2.5, 5))
collect_run(ws, 1000, label="Ten further points on the curve held back for acceptance", name="wasm_saturate_backtest", source="builder", build=lambda: labelled(5.0, 10))

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "wasm_saturate_training"), validation=ws.run(1000, "wasm_saturate_validation"), backtest=ws.run(1000, "wasm_saturate_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
def saturate(x, height, rate):
    return height * tanh(rate * x)


(height, rate), _ = curve_fit(saturate, ws.training[ws.inputs[0]].to_numpy(), ws.training[ws.outputs[0]].to_numpy(), p0=(3.0, 0.4))
HEIGHT, RATE = round(float(height), 1), round(float(rate), 1)
ws.params = {"height": HEIGHT, "rate": RATE}
print("fitted height", HEIGHT, "rate", RATE)
assert (HEIGHT, RATE) == (4.0, 0.5), "the fit did not recover the curve that made the data"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

HEIGHT, RATE = ws.params["height"], ws.params["rate"]


def saturate_model(saturate_in):
    return HEIGHT * tanh(RATE * saturate_in)


define_model(ws, model=1000, flavour="wasm", source=saturate_model, label="Least squares tanh curve, height and rate built in", constants={"HEIGHT": HEIGHT, "RATE": RATE})

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        predicted = evaluate_model(ws, 1000, split)[ws.outputs[0]]
        checks.append(
            {
                "split": split,
                "gate": split == "backtest",
                "output": ws.outputs[0],
                "name": "rmse",
                "value": evaluate_metrics(getattr(ws, split)[ws.outputs[0]], predicted)["rmse"],
                "limit": 0.5,
                "direction": "max",
            }
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

xs = [-3.0, -1.0, 0.0, 0.5, 1.0, 2.0, 6.0]
golden = {ws.inputs[0]: [*xs, None], "expect": ["ok"] * len(xs) + ["error"]}
package_model(ws, 1000, golden, target={"kind": "embedded", "language": "go"}, tolerance={ws.outputs[0]: {"absolute": 1e-12}})